# Syntopy Between Scales - Master Thesis

## Intial Data Preparation

In [1]:
import ee
import geemap
import pandas as pd
import geopandas as gpd

In [2]:
#  Initialise Earth Engine Project ID
try:
    ee.Initialize(project='syntopypredictors2026')

except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='syntopypredictors2026')

In [3]:
connectivity_test = ee.Number(1).add(2).getInfo()
print('connectivity test: ', connectivity_test)

connectivity test:  3


In [16]:
# Import Landsat Collection 2 Tier 1 Level 2 32-Day NDVI Composite
ndvi = ee.ImageCollection('LANDSAT/COMPOSITES/C02/T1_L2_32DAY_NDVI')
ndvi

In [27]:
# Initial date of interest (inclusive).
i_date = '2018-05-01'
# Final date of interest (exclusive).
f_date = '2018-07-01'

ndvi_mean = ndvi.filterDate(i_date, f_date).select('NDVI').mean()
ndvi_mean

In [22]:
dat = gpd.read_file('route_dat.gpkg', engine = 'fiona')
dat['lon'] = dat.geometry.x
dat['lat'] = dat.geometry.y
dat.head()

,RTENO,Year,geometry,lon,lat
0,12404002,2017,POINT (-111.4102 49.87324),-111.41020,49.87324
1,12404002,2018,POINT (-111.4102 49.87324),-111.41020,49.87324
2,12404002,2019,POINT (-111.4102 49.87324),-111.41020,49.87324
3,12404003,2017,POINT (-112.39699 49.08995),-112.39699,49.08995
4,12404004,2017,POINT (-113.56982 49.81509),-113.56982,49.81509


In [38]:
# Check unique active routes per year
active_counts = dat.groupby("Year")["RTENO"].nunique()
print(active_counts)

Year
2017    3021
2018    3002
2019    2891
Name: RTENO, dtype: int64


In [23]:
# Keep only the unique routes based on RTENO
unique_routes = dat.drop_duplicates(subset=['RTENO']).reset_index(drop=True)

print(f'Original rows with multi-year records: {len(dat)}')
print(f'Unique routes to extract: {len(unique_routes)}')

Original rows with multi-year records: 8914
Unique routes to extract: 3614


In [32]:
import time
import pandas as pd

chunk_size = 500
all_results = []

print('Starting batched local extraction...')

for i in range(0, len(unique_routes), chunk_size):
  chunk = unique_routes.iloc[i : i + chunk_size]

  # 1. Build features for the current chunk
  features = []
  for _, row in chunk.iterrows():
    point = ee.Geometry.Point([row['lon'], row['lat']])
    buff = point.buffer(3000)  # Your 3000m buffer
    feat = ee.Feature(buff, {'RTENO': row['RTENO']})
    features.append(feat)

  points_fc = ee.FeatureCollection(features)

  # 2. Run reduction on the small batch
  sampled_chunk = ndvi_mean.reduceRegions(
      collection=points_fc, reducer=ee.Reducer.mean(), scale=30
  )

  # 3. Safely fetch info for this small batch
  try:
    info = sampled_chunk.getInfo()
    for f in info['features']:
      all_results.append(f['properties'])
    print(
        f"Successfully processed rows {i} to"
        f" {min(i + chunk_size, len(unique_routes))}"
    )
  except Exception as e:
    print(f"Error on chunk starting at {i}: {e}")

  # Brief pause to be kind to the server
  time.sleep(1)

# Combine all chunks back into a single DataFrame
df_ndvi = pd.DataFrame(all_results)
data_final = dat.merge(df_ndvi[['RTENO', 'mean']], on='RTENO', how='left')
print('All routes successfully extracted and merged!')

Starting batched local extraction...
Successfully processed rows 0 to 500
Successfully processed rows 500 to 1000
Successfully processed rows 1000 to 1500
Successfully processed rows 1500 to 2000
Successfully processed rows 2000 to 2500
Successfully processed rows 2500 to 3000
Successfully processed rows 3000 to 3500
Successfully processed rows 3500 to 3614
All routes successfully extracted and merged!


In [33]:
# Check how many rows failed
print(data_final["mean"].isna().sum())

# Optional: Fill missing values with the median of the column or drop them explicitly
# df_ndvi['mean'] = df_ndvi['mean'].fillna(df_ndvi['mean'].median())

35


In [ ]:
# Impute the last few missing values using the state or regional median
data_final['mean'] = data_final.groupby('State')['mean'].transform(
    lambda x: x.fillna(x.median())
)

# Ultimate fallback just in case an entire state group is missing
df_ndvi['mean'] = df_ndvi['mean'].fillna(df_ndvi['mean'].median())

print(f"Remaining NaNs: {df_ndvi['mean'].isna().sum()}")  # Should be 0!

In [37]:
data_final.to_file("NDVI_mean_2018.gpkg", driver='GPKG')

In [ ]:
m = data_final.explore(
    column = 'mean',
    cmap='RdYlGn',
    legend=True,
    tooltip=['mean'],
    popup=True)
#m

In [87]:
years = ['2017', '2018', '2019']
final_yearly_datasets = {}

for yr in years:
  print(f'\n--- Processing Breeding Season NDVI for {yr} ---')

  # 1. Filter your master dataframe for just this year's active routes
  year_dat = dat[dat['Year'] == int(yr)].copy()
  print(f'Active routes for {yr}: {len(year_dat)}')

  # 2. Define year-specific breeding window (May 1 - July 1)
  i_date = f'{yr}-05-01'
  f_date = f'{yr}-07-01'

  ndvi_col = (
      ee.ImageCollection('LANDSAT/COMPOSITES/C02/T1_L2_32DAY_NDVI')
      .select('NDVI')
      .filterDate(i_date, f_date)
      .mean()
  )

  # 3. Batched extraction loop (chunk size 500)
  chunk_size = 500
  all_results = []

  for i in range(0, len(year_dat), chunk_size):
    chunk = year_dat.iloc[i : i + chunk_size]

    features = []
    for _, row in chunk.iterrows():
      point = ee.Geometry.Point([row['lon'], row['lat']])
      buff = point.buffer(300)  # 300m buffer
      feat = ee.Feature(buff, {'RTENO': row['RTENO']})
      features.append(feat)

    points_fc = ee.FeatureCollection(features)
    sampled_chunk = ndvi_col.reduceRegions(
        collection=points_fc, reducer=ee.Reducer.mean(), scale=30
    )

    try:
      info = sampled_chunk.getInfo()
      for f in info['features']:
        all_results.append(f['properties'])
      print(
          f'  Successfully processed rows {i} to'
          f' {min(i + chunk_size, len(year_dat))}'
      )
    except Exception as e:
      print(f'  Error on chunk starting at {i}: {e}')

    time.sleep(1)

  # 4. Convert to DataFrame and merge back to year_dat
  df_ndvi = pd.DataFrame(all_results)
  year_merged = year_dat.merge(
      df_ndvi[['RTENO', 'mean']], on='RTENO', how='left'
  )
  year_merged.rename(columns={'mean': 'NDVI'}, inplace=True)


--- Processing Breeding Season NDVI for 2017 ---
Active routes for 2017: 3021
  Successfully processed rows 0 to 500
  Successfully processed rows 500 to 1000
  Successfully processed rows 1000 to 1500
  Successfully processed rows 1500 to 2000
  Successfully processed rows 2000 to 2500
  Successfully processed rows 2500 to 3000
  Successfully processed rows 3000 to 3021

--- Processing Breeding Season NDVI for 2018 ---
Active routes for 2018: 3002
  Successfully processed rows 0 to 500
  Successfully processed rows 500 to 1000
  Successfully processed rows 1000 to 1500
  Successfully processed rows 1500 to 2000
  Successfully processed rows 2000 to 2500
  Successfully processed rows 2500 to 3000
  Successfully processed rows 3000 to 3002

--- Processing Breeding Season NDVI for 2019 ---
Active routes for 2019: 2891
  Successfully processed rows 0 to 500
  Successfully processed rows 500 to 1000
  Successfully processed rows 1000 to 1500
  Successfully processed rows 1500 to 2000
  Su

In [93]:
import time
import pandas as pd
import ee
from sklearn.neighbors import KNeighborsRegressor

years = ['2017', '2018', '2019']
final_yearly_datasets = {}  # Dictionary to hold each year

for yr in years:
  print(f'\n--- Processing Breeding Season NDVI for {yr} ---')

  # 1. Filter your master dataframe for just this year's active routes
  year_dat = dat[dat['Year'] == int(yr)].copy()
  print(f'Active routes for {yr}: {len(year_dat)}')

  # 2. Define year-specific breeding window (May 1 - July 1)
  i_date = f'{yr}-05-01'
  f_date = f'{yr}-07-01'

  ndvi_col = (
      ee.ImageCollection('LANDSAT/COMPOSITES/C02/T1_L2_32DAY_NDVI')
      .select('NDVI')
      .filterDate(i_date, f_date)
      .mean()
  )

  # 3. Batched extraction loop (chunk size 500)
  chunk_size = 500
  all_results = []

  for i in range(0, len(year_dat), chunk_size):
    chunk = year_dat.iloc[i : i + chunk_size]

    features = []
    for _, row in chunk.iterrows():
      point = ee.Geometry.Point([row['lon'], row['lat']])
      buff = point.buffer(300)  # 300m buffer
      feat = ee.Feature(buff, {'RTENO': row['RTENO']})
      features.append(feat)

    points_fc = ee.FeatureCollection(features)
    sampled_chunk = ndvi_col.reduceRegions(
        collection=points_fc, reducer=ee.Reducer.mean(), scale=30
    )

    try:
      info = sampled_chunk.getInfo()
      for f in info['features']:
        all_results.append(f['properties'])
      print(
          f'  Successfully processed rows {i} to'
          f' {min(i + chunk_size, len(year_dat))}'
      )
    except Exception as e:
      print(f'  Error on chunk starting at {i}: {e}')

    time.sleep(1)

  # 4. Convert to DataFrame and merge back to year_dat
  df_ndvi = pd.DataFrame(all_results)
  year_merged = year_dat.merge(
      df_ndvi[['RTENO', 'mean']], on='RTENO', how='left'
  )
  year_merged.rename(columns={'mean': 'NDVI'}, inplace=True)

  # 5. Apply Spatial KNN Interpolation for any missing values in this year
  valid_data = year_merged[year_merged['NDVI'].notna()]
  missing_data = year_merged[year_merged['NDVI'].isna()]

  if len(missing_data) > 0:
    print(
        f'  Interpolating {len(missing_data)} missing values via spatial'
        ' KNN...'
    )
    knn = KNeighborsRegressor(
        n_neighbors=5, weights='distance', metric='euclidean'
    )
    knn.fit(valid_data[['lon', 'lat']], valid_data['NDVI'])
    year_merged.loc[year_merged['NDVI'].isna(), 'NDVI'] = knn.predict(
        missing_data[['lon', 'lat']]
    )

  # ---> THIS LINE WAS MISSING IN YOUR SCRIPT <---
  # Save the finished year into our dictionary
  final_yearly_datasets[yr] = year_merged

print('\nAll years successfully processed!')

# ==========================================
# AFTER THE LOOP: Combine everything into one master dataframe
# ==========================================
df_all = pd.concat(final_yearly_datasets.values(), ignore_index=True)

# Verify all years are present now
print('Years present in master dataframe:', df_all['Year'].unique())


--- Processing Breeding Season NDVI for 2017 ---
Active routes for 2017: 3021
  Successfully processed rows 0 to 500
  Successfully processed rows 500 to 1000
  Successfully processed rows 1000 to 1500
  Successfully processed rows 1500 to 2000
  Successfully processed rows 2000 to 2500
  Successfully processed rows 2500 to 3000
  Successfully processed rows 3000 to 3021
  Interpolating 101 missing values via spatial KNN...

--- Processing Breeding Season NDVI for 2018 ---
Active routes for 2018: 3002
  Successfully processed rows 0 to 500
  Successfully processed rows 500 to 1000
  Successfully processed rows 1000 to 1500
  Successfully processed rows 1500 to 2000
  Successfully processed rows 2000 to 2500
  Successfully processed rows 2500 to 3000
  Successfully processed rows 3000 to 3002
  Interpolating 58 missing values via spatial KNN...

--- Processing Breeding Season NDVI for 2019 ---
Active routes for 2019: 2891
  Successfully processed rows 0 to 500
  Successfully processed 

In [95]:
df_all.head()

,RTENO,Year,geometry,lon,lat,NDVI
0,12404002,2017,POINT (-111.4102 49.87324),-111.41020,49.87324,0.394317
1,12404003,2017,POINT (-112.39699 49.08995),-112.39699,49.08995,0.522018
2,12404004,2017,POINT (-113.56982 49.81509),-113.56982,49.81509,0.618319
3,12404006,2017,POINT (-110.25869 50.74668),-110.25869,50.74668,0.483786
4,12404009,2017,POINT (-113.02852 50.60147),-113.02852,50.60147,0.586473


In [ ]:
import folium
import geopandas as gpd
from shapely.geometry import Point

# 1. Ensure your dataframe is a GeoDataFrame (if it isn't already)
if not isinstance(df_all, gpd.GeoDataFrame):
  geometry = [Point(xy) for xy in zip(df_all["lon"], df_all["lat"])]
  gdf = gpd.GeoDataFrame(df_all, geometry=geometry, crs="EPSG:4326")
else:
  gdf = df_all

# Get sorted list of available years
years = sorted(gdf["Year"].unique())

# 2. Initialize the map using the first year (e.g., 2017)
first_yr = years[0]
subset_first = gdf[gdf["Year"] == first_yr]

m = subset_first.explore(
    column="NDVI",
    cmap="RdYlGn",  # Yellow-Green colormap is perfect for vegetation/NDVI
    legend=True,
    name=str(first_yr),
    tooltip=["RTENO", "NDVI", "Year"],
    marker_kwds={"radius": 4, "fill": True},
)

# 3. Loop through the remaining years and add them as overlay layers onto the same map (m)
for yr in years[1:]:
  subset_yr = gdf[gdf["Year"] == yr]
  subset_yr.explore(
      m=m,  # Pass the existing map object here
      column="NDVI",
      cmap="RdYlGn",
      legend=False,  # Skip duplicate legends to keep the UI clean
      name=str(yr),
      tooltip=["RTENO", "NDVI", "Year"],
      marker_kwds={"radius": 4, "fill": True},
  )

# 4. Add the interactive Layer Control box (top-right corner of the map)
folium.LayerControl(collapsed=False).add_to(m)

# Display the map in your Jupyter Notebook / Lab
#m